# Tensor Practices

1. 创建 Tensor；
2. 查看 shape、dtype、device；
3. reshape、索引和切片；
4. Tensor 运算与广播；
5. 矩阵乘法；
6. 聚合统计；
7. NumPy 与 Tensor 转换；
8. CPU/GPU 转移；
9. `requires_grad` 和简单梯度。


In [2]:
import numpy as np
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.3.1+cu118
CUDA available: True


## 1. 创建 Tensor
    - ‘torch.tensor([])'
    - 'torch.ones()/zeros()/rand()/randn()/arange()'


In [2]:
# 从 Python 列表创建
integer_tensor = torch.tensor([1, 2, 3, 4])
float_tensor = torch.tensor([1.0, 2.0, 3.0])

# 创建特殊 Tensor（参数/shape元组）
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
random_uniform = torch.rand(2, 3)   # 从均匀分布 U(0, 1) 中采样,生成的值范围是 [0, 1) 之间的随机数
random_normal = torch.randn(2, 3)   # 从标准正态分布 N(0, 1)中采样, 生成的值理论上可以是任意实数（正负都有）
sequence = torch.arange(0, 12)      # [0, 12) 同 arange(12), arange(0, 12, 1)

print("integer_tensor:", integer_tensor)
print("float_tensor:", float_tensor)
print("zeros:", zeros)
print("ones:", ones)
print("random_uniform:", random_uniform)
print("random_normal:", random_normal)
print("sequence:", sequence)


integer_tensor: tensor([1, 2, 3, 4])
float_tensor: tensor([1., 2., 3.])
zeros: tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones: tensor([[1., 1., 1.],
        [1., 1., 1.]])
random_uniform: tensor([[0.5310, 0.9416, 0.9124],
        [0.6184, 0.2597, 0.0657]])
random_normal: tensor([[ 0.3307,  0.4274, -0.4467],
        [-1.3482,  0.3004,  0.1839]])
sequence: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


## 2. shape、dtype、device、ndim 和 numel


In [3]:
x = torch.randn(2, 3, 4)

print("x shape:", x.shape)
print("x dtype:", x.dtype)
print("x device:", x.device)
print("x ndim:", x.ndim)
print("x numel:", x.numel())    # 张量 x 中元素总个数


x shape: torch.Size([2, 3, 4])
x dtype: torch.float32
x device: cpu
x ndim: 3
x numel: 24


## 3. 改变数据类型


In [4]:
x_int = torch.tensor([1, 2, 3])
x_float = x_int.float()
x_long = x_float.long() # 截断

print(x_int, x_int.dtype)
print(x_float, x_float.dtype)
print(x_long, x_long.dtype)


tensor([1, 2, 3]) torch.int64
tensor([1., 2., 3.]) torch.float32
tensor([1, 2, 3]) torch.int64


## 4. reshape 与 view
    - 'x.reshape()'
    - 'x.view()'


In [5]:
x = torch.arange(12)

x_3_by_4 = x.reshape(3, 4)
x_2_by_6 = x.view(2, 6)         # view()要求张量必须是连续的,只使用torch.arange(12)
x_inferred = x.reshape(3, -1)

print("original:", x)
print("reshape(3, 4):", x_3_by_4)
print("view(2, 6):", x_2_by_6)
print("reshape(3, -1):", x_inferred)


original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
reshape(3, 4): tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
view(2, 6): tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
reshape(3, -1): tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


`-1` 表示让 PyTorch 根据元素总数自动推断该维度。改变形状前后，元素总数必须保持不变。


## 5. 索引x[,]与切片(:)
### PyTorch 张量切片语法规则总结  -> 依旧 range 语法 * 3 （rang, arange, :)

| 写法 | 含义 |
|---|---|
| `:` | 这个维度全选，什么都不筛 |
| `a:b` | 从索引 a 到 b（含a不含b） |
| `:b` | 从开头到索引 b（不含b），等价于 `0:b` |
| `a:` | 从索引 a 到末尾，等价于 `a:末尾` |
| `a:b:step` | 从 a 到 b，每隔 step 取一个 |
### 索引
 '-1' 表示从后往前数
 当张量只有一维时，默认为列向量此时:x[0],x[1],x[2]

In [6]:
x = torch.arange(12).reshape(3, 4)
print("x:", x)

print("第 0 行:", x[0])
print("第 1 列:", x[:, 1])
print("前两行、前两列:", x[:2, :2])
print("最后一个元素:", x[-1, -1])


x: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
第 0 行: tensor([0, 1, 2, 3])
第 1 列: tensor([1, 5, 9])
前两行、前两列: tensor([[0, 1],
        [4, 5]])
最后一个元素: tensor(11)


## 6. 按条件筛选

- 'x[x > 4]'


In [7]:
x = torch.tensor([1, 5, 2, 8, 3, 10])
mask = x > 4          # bool 类型 x shape 的 tensor
selected = x[mask]    # x[x > 4]

print("mask:", mask)
print("selected:", selected)


mask: tensor([False,  True, False,  True, False,  True])
selected: tensor([ 5,  8, 10])


## 7. 基础运算


In [8]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b:", a + b)
print("a - b:", a - b)
print("a * b（逐元素乘法）:", a * b)
print("a / b:", a / b)
print("a ** 2:", a ** 2)


a + b: tensor([5., 7., 9.])
a - b: tensor([-3., -3., -3.])
a * b（逐元素乘法）: tensor([ 4., 10., 18.])
a / b: tensor([0.2500, 0.4000, 0.5000])
a ** 2: tensor([1., 4., 9.])


## 8. 广播机制(小扩大）


In [9]:
matrix = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])
row = torch.tensor([10, 20, 30])

result = matrix + row

print("matrix shape:", matrix.shape)
print("row shape:", row.shape)
print("result:", result)


matrix shape: torch.Size([2, 3])
row shape: torch.Size([3])
result: tensor([[11, 22, 33],
        [14, 25, 36]])


## 9. 矩阵乘法


In [10]:
A = torch.randn(3, 4)
B = torch.randn(4, 2)

C1 = A @ B
C2 = torch.matmul(A, B)

print("A shape:", A.shape)
print("B shape:", B.shape)
print("C1 shape:", C1.shape)
print("C1 与 C2 是否相同:", torch.allclose(C1, C2))


A shape: torch.Size([3, 4])
B shape: torch.Size([4, 2])
C1 shape: torch.Size([3, 2])
C1 与 C2 是否相同: True


`A @ B` 是矩阵乘法，而 `A * B` 是逐元素乘法。


## 10. 拼接 Tensor(要求拼接维度相同）
- 'torch.cat([a, b], dim=)'
- torch.stack([a, b], dim=)'


In [11]:
a = torch.ones(2, 3)
b = torch.zeros(2, 3)

concat_dim0 = torch.cat([a, b], dim=0)  # 在第0维（行方向） 上拼接，相当于把 b 接到 a 下面
concat_dim1 = torch.cat([a, b], dim=1)  # 在第1维（列方向） 上拼接，相当于把 b 接到 a 右边：
stacked = torch.stack([a, b], dim=0)    # stack 会新建一个维度, 把 a 和 b 当作两个"元素"，堆成一个新的维度

print("cat dim=0 shape:", concat_dim0.shape)
print("cat dim=1 shape:", concat_dim1.shape)
print("stack shape:", stacked.shape)


cat dim=0 shape: torch.Size([4, 3])
cat dim=1 shape: torch.Size([2, 6])
stack shape: torch.Size([2, 2, 3])


## 11. 聚合统计
- 'sum(dim=)'
- 'mean(dim=)'
- 'max(dim=)'

In [3]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])

print("sum:", x.sum())                  # 无dim参照则为整体
print("mean:", x.mean())
print("max:", x.max())
print("每列均值:", x.mean(dim=0))       # dim 从一个维度统计(每行dim=1, 每列dim=0)
print("每行最大值:", x.max(dim=1).values)
print("每行最大值位置:", x.max(dim=1).indices)


sum: tensor(21.)
mean: tensor(3.5000)
max: tensor(6.)
每列均值: tensor([2.5000, 3.5000, 4.5000])
每行最大值: tensor([3., 6.])
每行最大值位置: tensor([2, 2])


## 12. NumPy 与 Tensor 转换


In [4]:
numpy_array = np.array([1.0, 2.0, 3.0], dtype=np.float32)
tensor_from_numpy = torch.from_numpy(numpy_array)
back_to_numpy = tensor_from_numpy.numpy()

print("numpy_array:", numpy_array, type(numpy_array))
print("tensor_from_numpy:", tensor_from_numpy, type(tensor_from_numpy))
print("back_to_numpy:", back_to_numpy, type(back_to_numpy))


numpy_array: [1. 2. 3.] <class 'numpy.ndarray'>
tensor_from_numpy: tensor([1., 2., 3.]) <class 'torch.Tensor'>
back_to_numpy: [1. 2. 3.] <class 'numpy.ndarray'>


注意：CPU 上由 `torch.from_numpy()` 创建的 Tensor 与原 NumPy 数组可能共享内存。修改其中一个，另一个也可能改变。


In [5]:
numpy_array[0] = 100
print("modified numpy_array:", numpy_array)
print("shared tensor:", tensor_from_numpy)


modified numpy_array: [100.   2.   3.]
shared tensor: tensor([100.,   2.,   3.])


## 13. CPU 与 GPU 转移


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(3, 4)
print("before:", x.device)

x = x.to(device)
print("after:", x.device)

# GPU Tensor 转 NumPy 前必须先移回 CPU
x_numpy = x.detach().cpu().numpy()
print("NumPy shape:", x_numpy.shape)


before: cpu
after: cuda:0
NumPy shape: (3, 4)


模型参数和输入数据必须位于同一个设备，例如都在 CPU，或者都在同一块 GPU。


## 14. requires_grad 与自动求导

backward() 只认标量，.item() 只认单元素张量——遇到向量，该求和求和，该直接打印就直接打印。

In [7]:
x = torch.tensor(2.0, requires_grad=True)   # torch.tensor(2.0) 创建的是一个 0维张量(标量）, 没有"行"的概念
y = x ** 2 + 3 * x + 1  

y.backward()        # 只有当输出是一个标量（一个数）时，才能不带参数地直接调用 .backward()。

print("y:", y.item())           # .item()：把只含一个数值的张量(标量）转换成普通 Python 数字，方便打印
print("dy/dx:", x.grad.item())  # x.grad：存储的就是 PyTorch 自动计算出的梯度 dy/dx


x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x ** 2 + x * 3 + 1

loss = y.sum()
print(loss.item())
loss.backward()

print(x.grad)


y: 11.0
dy/dx: 7.0


因为：

\[
y=x^2+3x+1
\]

所以：

\[
rac{dy}{dx}=2x+3
\]

当 `x=2` 时，梯度等于 `7`。

## 15. 模拟一批图像数据


In [8]:
# 定义批大小
batch_size = 32
# 生成经典 MNIST 数据集 图像数据 28×28 的灰度图
images = torch.randn(batch_size, 1, 28, 28) # 一批32张图片, 1代表灰度图（黑白）, 图片高28像素, 宽28像素
# 生成模拟标签数据
labels = torch.randint(low=0, high=10, size=(batch_size,))  # 生成 [0, 10) 区间的随机整数,32张图片，每张图对应一个标签

# 每张 1×28×28 的图片"拍扁"成一个长向量
flattened_images = images.reshape(batch_size, -1)   # 一张图的所有像素展平后排成一行

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("flattened images shape:", flattened_images.shape)


images shape: torch.Size([32, 1, 28, 28])
labels shape: torch.Size([32])
flattened images shape: torch.Size([32, 784])


FashionMNIST 的单张图片形状是 `[1, 28, 28]`。一批 32 张图片的形状是 `[32, 1, 28, 28]`。经过展平后变成 `[32, 784]`，可以送入全连接层。
